# Exercices · Intervalles de confiance

Exercices réalisés pendant le bootcamp Data Essentials (Jedha, 2026). Les jeux de données ont été fournis par le bootcamp et ne sont pas inclus dans le dépôt.

**Rappel** : un intervalle de confiance à 95 % donne une fourchette de valeurs plausibles pour un paramètre de la population (une moyenne, une proportion) à partir d'un échantillon. Si l'on répétait l'échantillonnage un grand nombre de fois, 95 % des intervalles construits de cette façon contiendraient la vraie valeur.

In [ ]:
import math

import numpy as np
import pandas as pd
from scipy import stats

# Dossier contenant les fichiers de données fournis par le bootcamp
# Sur Google Colab : téléverser les fichiers puis utiliser DATA_DIR = "/content/"
DATA_DIR = "data/"

### Fonction utilisée pour les moyennes

Quand l'écart-type de la population est inconnu (ce qui est presque toujours le cas), la bonne référence est la **loi de Student** à n − 1 degrés de liberté. Pour un grand échantillon, elle se confond avec la loi normale : 1,98 contre 1,96 pour n = 102. Pour un petit échantillon, la différence compte : 2,04 pour n = 31.

In [ ]:
def intervalle_moyenne(serie, niveau=0.95):
    """Intervalle de confiance d'une moyenne (loi de Student, écart-type inconnu)."""
    n = serie.count()
    moyenne = serie.mean()
    erreur_standard = serie.std() / math.sqrt(n)
    t_critique = stats.t.ppf((1 + niveau) / 2, df=n - 1)
    marge = t_critique * erreur_standard
    return moyenne - marge, moyenne + marge

## Exercice 1 · Statistiques de base (California Housing)

Premières manipulations sur le jeu de données immobilier de Californie fourni par scikit-learn.

In [ ]:
from sklearn.datasets import fetch_california_housing

cali = fetch_california_housing()
cali_houses = pd.DataFrame(data=cali.data, columns=cali.feature_names)

print(f"{cali_houses.shape[0]} quartiers, {cali_houses.shape[1]} variables")
cali_houses.head()

In [ ]:
print("Âge moyen des logements :", round(cali_houses["HouseAge"].mean(), 2), "ans")
print("Écart-type de la population par quartier :", round(cali_houses["Population"].std(), 1))
cali_houses.describe().round(2)

## Exercice 2 · Facebook

**Question :** estimer, avec un niveau de confiance de 95 %, le temps moyen passé sur Facebook mobile par les utilisateurs.

In [ ]:
hrs_fb = pd.read_excel(DATA_DIR + "Hours Spent on Facebook.xlsx")
colonne_fb = "Hours spent on Facebook on Mobile"

hrs_fb[colonne_fb].describe().round(2)

In [ ]:
bas, haut = intervalle_moyenne(hrs_fb[colonne_fb])
print(f"Moyenne observée : {hrs_fb[colonne_fb].mean():.2f} h")
print(f"IC à 95 % : [{bas:.2f} h ; {haut:.2f} h]")

**Conclusion :** avec 95 % de confiance, le temps moyen passé sur Facebook mobile se situe entre environ 4,26 et 5,43 heures.

## Exercice 3 · Google Ads

**Question :** estimer, avec un niveau de confiance de 95 %, la dépense mensuelle moyenne en Google Ads.

*Correction :* la version initiale utilisait 2,228 pour la borne basse et 1,96 pour la borne haute, ce qui donnait un intervalle asymétrique. De plus, 2,228 correspond à 10 degrés de liberté, alors qu'avec n = 31 il en faut 30 (valeur critique 2,042).

In [ ]:
ad_ggl = pd.read_excel(DATA_DIR + "Monthly spending on adwords.xlsx")
ad_ggl["Monthly spending"].describe().round(2)

In [ ]:
n_ad = ad_ggl["Monthly spending"].count()
print(f"Valeur critique de Student ({n_ad - 1} ddl) : {stats.t.ppf(0.975, df=n_ad - 1):.3f}")

bas, haut = intervalle_moyenne(ad_ggl["Monthly spending"])
print(f"Moyenne observée : {ad_ggl['Monthly spending'].mean():.2f}")
print(f"IC à 95 % : [{bas:.2f} ; {haut:.2f}]")

**Conclusion :** avec 95 % de confiance, la dépense mensuelle moyenne est comprise entre environ 398,6 et 584,9.

## Exercice 4 · Nintendo

**Question :** combien de villes faut-il étudier pour estimer le nombre moyen de Switch vendues avec une marge d'erreur de 5 000 unités, au niveau de confiance de 95 % ?

La taille d'échantillon se calcule avec n = (z × σ / E)², où σ est estimé par l'écart-type des 20 villes observées. Le résultat s'arrondit **toujours à l'entier supérieur** : arrondir à l'inférieur donnerait une marge légèrement supérieure à 5 000.

In [ ]:
nin = pd.read_excel(DATA_DIR + "Number of switch sold.xlsx")
nin["Number of sales"].describe().round(0)

In [ ]:
z = stats.norm.ppf(0.975)   # 1,96
marge_voulue = 5000
ecart_type_nin = nin["Number of sales"].std()

n_calcule = (z * ecart_type_nin / marge_voulue) ** 2
n_necessaire = math.ceil(n_calcule)

print(f"Taille calculée : {n_calcule:.1f}")
print(f"Nombre de villes nécessaires : {n_necessaire}")

**Conclusion :** il faut étudier au moins **124 villes**.

## Exercice 5 · Apple

**Question :** estimer, avec un niveau de confiance de 95 %, la proportion de clients satisfaits de la nouvelle sortie Apple.

*Correction :* la version initiale fixait n = 40, alors que le fichier contient 41 réponses (15 satisfaits + 26 non satisfaits). La taille de l'échantillon est maintenant lue directement dans les données, et l'intervalle est calculé jusqu'au bout.

In [ ]:
apl = pd.read_excel(DATA_DIR + "Satisfaction apple release.xlsx")

n_apl = len(apl)
k = (apl["Satisfied customer"] == 1).sum()
p = k / n_apl

print(f"{k} clients satisfaits sur {n_apl}, soit p = {p:.3f}")

Pour une proportion, l'approximation par la loi normale est valable si n × p et n × (1 − p) sont tous deux supérieurs à 5 : c'est le cas ici (15 et 26).

In [ ]:
z = stats.norm.ppf(0.975)
erreur_standard = math.sqrt(p * (1 - p) / n_apl)
marge = z * erreur_standard

print(f"IC à 95 % : [{p - marge:.3f} ; {p + marge:.3f}]")

**Conclusion :** avec 95 % de confiance, la proportion de clients satisfaits se situe entre environ 22 % et 51 %. L'intervalle est large car l'échantillon est petit (41 réponses) : il faudrait interroger plus de clients pour une estimation précise.